In [4]:
import pandas as pd
import dash
from dash import dcc, html, Output, Input, State
import dash_cytoscape as cyto
import dash_bootstrap_components as dbc
from flask import Flask, Response
from bokeh.plotting import figure
from bokeh.models import RangeTool, ColumnDataSource, CustomJS
from bokeh.embed import file_html
from bokeh.resources import INLINE
from bokeh.layouts import column

# ----------------------------
# Load CSV Data
# ----------------------------
df = pd.read_csv(
    "/home/apowers/Projects/roc/experiments/2025.04.22-21.26.35-bokeh-learning/data.csv"
)
df["transaction_date"] = pd.to_datetime(df["transaction_date"], format="%d/%m/%Y")
min_date = df["transaction_date"].min()
df["date_numeric"] = (df["transaction_date"] - min_date).dt.days

# ----------------------------
# Flask & Dash Setup
# ----------------------------
server = Flask(__name__)

# Load an attractive Bootswatch theme, e.g., FLATLY
external_stylesheets = [dbc.themes.FLATLY]
app = dash.Dash(__name__, server=server, external_stylesheets=external_stylesheets)


# ----------------------------
# Helper Function: Generate Cytoscape Elements
# ----------------------------
def generate_cytoscape_elements(start_date, end_date):
    filtered_df = df[(df["transaction_date"] >= start_date) & (df["transaction_date"] <= end_date)]
    nodes, edges = [], []
    unique_nodes = set()
    for _, row in filtered_df.iterrows():
        src = row["source"]
        tgt = row["target"]
        transaction_value = row["transaction_value"]
        transaction_type = row["transaction_type"]
        if src not in unique_nodes:
            nodes.append({"data": {"id": src, "label": src}})
            unique_nodes.add(src)
        if tgt not in unique_nodes:
            nodes.append({"data": {"id": tgt, "label": tgt}})
            unique_nodes.add(tgt)
        edges.append(
            {
                "data": {
                    "source": src,
                    "target": tgt,
                    "label": f"${transaction_value} ({transaction_type})",
                }
            }
        )
    return nodes + edges


# ----------------------------
# Bokeh RangeTool Plot Route
# ----------------------------
@server.route("/bokeh_range")
def bokeh_range():
    # Group data by transaction_date and calculate frequency
    grouped_data = df.groupby("transaction_date").size().reset_index(name="count")
    grouped_data["date_numeric"] = (grouped_data["transaction_date"] - min_date).dt.days

    source = ColumnDataSource(data={"x": grouped_data["date_numeric"], "y": grouped_data["count"]})

    # Main bar chart
    p = figure(
        height=0,
        width=0,
        tools="xpan",
        toolbar_location=None,
        x_range=(grouped_data["date_numeric"].min(), grouped_data["date_numeric"].max()),
        title="Transaction Frequency Over Time",
    )
    p.vbar(x="x", top="y", source=source, width=0.9, color="navy")
    p.title.text_font_size = "18pt"
    p.xaxis.axis_label = "Days Since " + min_date.date().isoformat()
    p.yaxis.axis_label = "Transaction Frequency"

    # RangeTool selector
    select = figure(
        height=150,
        width=1200,
        y_range=p.y_range,
        x_range=(grouped_data["date_numeric"].min(), grouped_data["date_numeric"].max()),
        tools="",
        toolbar_location=None,
        background_fill_color="#efefef",
    )
    select.vbar(x="x", top="y", source=source, width=0.9, color="gray")

    range_tool = RangeTool(x_range=p.x_range)
    range_tool.overlay.fill_color = "navy"
    range_tool.overlay.fill_alpha = 0.2
    select.add_tools(range_tool)

    callback = CustomJS(
        code="""
        console.log("Bokeh update: start=", cb_obj.start, " end=", cb_obj.end);
        window.parent.postMessage({
            type: 'range_update', 
            start: cb_obj.start, 
            end: cb_obj.end
        }, "*");
    """
    )
    p.x_range.js_on_change("start", callback)
    p.x_range.js_on_change("end", callback)

    layout_ = column(p, select)
    bokeh_html = file_html(layout_, INLINE, "Bokeh RangeTool Demo")
    return Response(bokeh_html, mimetype="text/html")


# ----------------------------
# Cytoscape Styling
# ----------------------------
cyto_stylesheet = [
    {
        "selector": "node",
        "style": {
            "background-color": "#0074D9",
            "label": "data(label)",
            "color": "white",
            "text-valign": "center",
            "text-halign": "center",
            "font-size": "12px",
            "width": "50px",
            "height": "50px",
        },
    },
    {
        "selector": "edge",
        "style": {
            "line-color": "#FF4136",
            "target-arrow-color": "#FF4136",
            "target-arrow-shape": "triangle",
            "curve-style": "bezier",
            "label": "data(label)",
            "font-size": "10px",
            "text-rotation": "autorotate",
            "color": "#000",
        },
    },
]

# ----------------------------
# Dash Layout: Horizontal Row for Charts
# ----------------------------
app.layout = dbc.Container(
    [
        dbc.Row(
            dbc.Col(
                html.H1("Transaction Network Filtered by Date Range", className="my-4"),
                width=12,
                className="text-center",
            )
        ),
        dbc.Row(
            [
                dbc.Col(
                    cyto.Cytoscape(
                        id="cytoscape-graph",
                        elements=[],  # Will update via callback.
                        layout={"name": "circle"},
                        stylesheet=cyto_stylesheet,
                        style={
                            "width": "1200px",
                            "height": "800px",
                            "border": "1px solid #ccc",
                            "margin": "0 auto",
                        },
                    ),
                    width="auto",
                    className="d-flex justify-content-center",
                )
            ],
            className="mb-4 justify-content-center",
        ),
        dbc.Row(
            [
                dbc.Col(
                    html.Iframe(
                        src="/bokeh_range",
                        style={
                            "width": "1200px",
                            "height": "200px",
                            "border": "none",
                            "display": "block",
                            "margin": "0 auto",
                        },
                    ),
                    width="auto",
                    className="d-flex justify-content-center",
                )
            ],
            className="mb-4 justify-content-center",
        ),
        dbc.Row(
            dbc.Col(
                dcc.Store(
                    id="range-store",
                    data={
                        "start": min_date.isoformat(),
                        "end": df["transaction_date"].max().isoformat(),
                    },
                ),
                width=12,
                className="text-center",
            )
        ),
        dbc.Row(
            dbc.Col(
                dcc.Interval(id="interval", interval=200, n_intervals=0),
                width=12,
                className="text-center",
            )
        ),
        dbc.Row(
            dbc.Col(
                html.Div(id="debug-output", className="font-weight-bold mt-3"),
                width=12,
                className="text-center",
            )
        ),
    ],
    fluid=True,
    style={"font-family": "Arial"},
)

# ----------------------------
# Clientside Callback: Update Store with New Date Range
# ----------------------------
baseline_str = min_date.isoformat()
app.clientside_callback(
    f"""
    function(n_intervals, currentData) {{
        if (window.latestRange != null && typeof window.latestRange.start !== 'undefined') {{
            const newRange = window.latestRange;
            window.latestRange = null;
            const baseline = new Date("{baseline_str}");
            const startDate = new Date(baseline.getTime() + newRange.start * 86400000);
            const endDate = new Date(baseline.getTime() + newRange.end * 86400000);
            console.log("Updating store with", {{start: startDate.toISOString(), end: endDate.toISOString()}});
            return {{start: startDate.toISOString(), end: endDate.toISOString()}};
        }}
        return currentData;
    }}
    """,
    Output("range-store", "data"),
    Input("interval", "n_intervals"),
    State("range-store", "data"),
)


# ----------------------------
# Server Callback: Update Cytoscape Graph & Debug Info
# ----------------------------
@app.callback(
    [Output("cytoscape-graph", "elements"), Output("debug-output", "children")],
    [Input("range-store", "data")],
)
def update_cytoscape(range_data):
    start_date = pd.to_datetime(range_data["start"]).tz_localize(None)
    end_date = pd.to_datetime(range_data["end"]).tz_localize(None)
    elements = generate_cytoscape_elements(start_date, end_date)
    debug_text = (
        f"Updating Cytoscape: Date Range = ({start_date.date()} to {end_date.date()}), "
        f"Node/Edge Count = {len(elements)}"
    )
    print(debug_text)
    return elements, debug_text


# ----------------------------
# Run the App
# ----------------------------
if __name__ == "__main__":
    app.run(debug=True)